# Sizing & tuning

How few time steps can you get away with? This page covers the two levers that
control the size of the aggregation — **number of periods** and **segments per
period** — and how to let tsam pick them for you.

In [ ]:
import pandas as pd
import plotly.express as px
import plotly.io as pio

import tsam

pio.renderers.default = "notebook_connected"

raw = pd.read_csv("testdata.csv", index_col=0, parse_dates=True)
data = raw.loc["2010-01-01":"2010-02-11"]  # six weeks of hourly data

## Lever 1 — number of periods

More clusters means more accuracy but less compression. Sweep a few values and
watch the error fall off:

In [ ]:
rows = []
for k in [2, 4, 6, 8, 12, 16]:
    r = tsam.aggregate(data, n_clusters=k, period_duration="1D")
    rows.append(
        {
            "n_clusters": k,
            "timesteps": k * r.n_timesteps_per_period,
            "rmse": float(r.accuracy.rmse.mean()),
        }
    )
sweep = pd.DataFrame(rows)
px.line(
    sweep,
    x="timesteps",
    y="rmse",
    markers=True,
    title="Accuracy vs. size — number of periods",
)

## Lever 2 — segments per period

Segmentation merges adjacent hours into a few variable-length segments, lowering
the resolution *within* each day. The two levers combine: `n_clusters` × `n_segments`
is your time-step budget.

In [ ]:
from tsam import SegmentConfig

segmented = tsam.aggregate(
    data,
    n_clusters=6,
    period_duration="1D",
    segments=SegmentConfig(n_segments=6),
)
segmented.plot.segment_durations()

## Let tsam choose

Picking by hand is fine for one dataset. Give
[`find_optimal_combination`](../api/tuning.md) a target **data reduction** and it
searches period/segment combinations for the most accurate one that hits it:

In [ ]:
from tsam.tuning import find_optimal_combination

best = find_optimal_combination(
    data,
    data_reduction=0.1,
    period_duration="1D",
    n_jobs=1,
    show_progress=False,
)
print(
    f"{best.n_clusters} periods x {best.n_segments} segments  ->  RMSE {best.rmse:.4f}"
)

## The whole trade-off curve

[`find_pareto_front`](../api/tuning.md) maps the accuracy-vs-size frontier so you
can see the knee and choose deliberately:

In [ ]:
from tsam.tuning import find_pareto_front

pareto = find_pareto_front(
    data,
    period_duration="1D",
    timesteps=range(12, 84, 12),
    n_jobs=1,
    show_progress=False,
)
pareto.summary

In [ ]:
pareto.plot()

The `best.best_result` (and every point on the Pareto front) is a regular
aggregation result — feed it straight to your model. See the
[Optimization workflow](optimization_workflow.ipynb).